# TicketIQ - End-to-End ML Walkthrough
Data collection -> preprocessing -> features -> classical ML -> PyTorch -> (optional) fine-tuning -> RAG agent. Run from the project root's `notebooks/` folder.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.config import *
from src.data import collect, preprocess
sns.set_theme(style='whitegrid')

## 1. Data collection

In [ ]:
collect.write_knowledge_base()
raw = collect.load_or_generate(4000)
raw.head()

## 2. Preprocessing & EDA

In [ ]:
df = preprocess.preprocess(raw)
train, test = preprocess.split(df)
print(df.shape, df.isna().sum().sum())
df[['text','text_clean','n_words','n_urgent']].head()

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(15,4))
sns.countplot(data=df, x='category', ax=ax[0])
sns.boxplot(data=df, x='category', y='resolution_hours', ax=ax[1])
sns.heatmap(df[['n_chars','n_words','n_exclaim','n_urgent','tenure_months','prior_tickets','resolution_hours']].corr(), annot=True, fmt='.2f', ax=ax[2])
plt.tight_layout()

## 3. Feature engineering (TF-IDF + numeric + one-hot)

In [ ]:
from src.features.engineering import build_preprocessor
prep = build_preprocessor().fit(train)
Xtr, Xte = prep.transform(train), prep.transform(test)
Xtr.shape

## 4. Classical models: LogReg, KNN, SVM, Decision Tree, Random Forest, Boosting

In [ ]:
from src.models import classical
clf_res, clf_models = classical.train_classifiers(Xtr, train.category, Xte, test.category)
clf_res

In [ ]:
sns.barplot(data=clf_res, y='model', x='f1_macro'); plt.xlim(0.5,1)

## 5. Regression: predict resolution hours

In [ ]:
reg_res, reg_models = classical.train_regressors(Xtr, train.resolution_hours, Xte, test.resolution_hours)
reg_res

## 6. Deep learning with PyTorch

In [ ]:
from src.models import deep
classes = sorted(train.category.unique())
metrics, hist = deep.train_mlp(Xtr, train.category, Xte, test.category, classes)
plt.plot(hist['train_loss'], label='train'); plt.plot(hist['val_loss'], label='val'); plt.legend()

## 7. Fine-tuning DistilBERT (optional - slow on CPU)

In [ ]:
# from src.models.finetune import finetune_distilbert
# finetune_distilbert(train, test, epochs=2, sample=2000)

## 8. Full pipeline + RAG agent (LangGraph + Groq + Pinecone)
Run `python -m src.pipeline` first to save models.

In [ ]:
from src.pipeline import run
run(skip_deep=True)

In [ ]:
from src.rag.graph import handle_ticket
r = handle_ticket('URGENT: I was charged twice for my Pro plan!', channel='email')
{k: r[k] for k in ['category','confidence','eta_hours','escalate','reason','answer']}

## 9. Observability
Per-node latency events are in `reports/logs/events.jsonl`; LangSmith traces appear if `LANGSMITH_API_KEY` is set.

In [ ]:
from src.observability import read_events
read_events().tail()